# Mask2Former Training — Lane Detection

**Architecture:** Mask2Former + Swin-Small backbone  
**Dataset:** Lane Detection (5 classes, 1610 images)  
**Strategy:** Gradual freezing (3 phases) + CosineAnnealingWarmRestarts  
**Tracking:** MLflow → DagsHub  
**Code sync:** GitHub (`git pull` in Colab Cell 4)

---
**Before running:** Runtime → GPU (T4). Colab secrets: `GITHUB_TOKEN`, `ROBOFLOW_API_KEY`, `HF_TOKEN`, MLflow.

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'torch: {torch.__version__}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers accelerate
!pip install -q mlflow pymongo
!pip install -q albumentations pycocotools
!pip install -q roboflow dvc python-dotenv
print('Dependencies installed')

In [ ]:
# ── Cell 3: Load secrets from Colab ─────────────────────────────────────────
# Add these in Colab: left sidebar → 🔑 Secrets
#   ROBOFLOW_API_KEY, HF_TOKEN, MONGO_URI, MLFLOW_TRACKING_URI
import os
from google.colab import userdata

os.environ['ROBOFLOW_API_KEY']    = userdata.get('ROBOFLOW_API_KEY')
os.environ['ROBOFLOW_WORKSPACE']  = 'test-mfeql'
os.environ['ROBOFLOW_PROJECT']    = 'lane-detection-segmentation-edyqp-fibkz'
os.environ['ROBOFLOW_VERSION']    = '1'
os.environ['HF_TOKEN']            = userdata.get('HF_TOKEN')
os.environ['HF_REPO_ID']          = 'srnortw/mask2former-lane-seg'
os.environ['MONGO_URI']           = userdata.get('MONGO_URI')
os.environ['MONGO_DB_NAME']       = 'mask2former'
os.environ['MONGO_COLLECTION_PREDICTIONS'] = 'predictions'
os.environ['MONGO_COLLECTION_DRIFT']       = 'drift_reports'
os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
os.environ['MLFLOW_TRACKING_USERNAME']  = 'srnortw'
os.environ['MLFLOW_TRACKING_PASSWORD']  = userdata.get('MLFLOW_TRACKING_PASSWORD')
os.environ['MLFLOW_EXPERIMENT_NAME']    = 'mask2former-swin'
os.environ['GITHUB_TOKEN']        = userdata.get('GITHUB_TOKEN')
os.environ['GITHUB_REPO']         = 'srnortw/mask2former'
print('Secrets loaded')

In [ ]:
# ── Cell 4: Clone repo (git) + download data (Roboflow) ─────────────────────
import os

REPO = '/content/mask2former'
token = os.environ.get('GITHUB_TOKEN') or userdata.get('GITHUB_TOKEN')

import subprocess
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', f'https://{token}@github.com/srnortw/mask2former.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

%cd {REPO}

ann = os.path.join(REPO, 'data/raw/train/_annotations.coco.json')
if os.path.isfile(ann):
    print('data/raw already present — skipping Roboflow download')
else:
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
    project = rf.workspace('test-mfeql').project('lane-detection-segmentation-edyqp-fibkz')
    dataset = project.version(1).download('coco-segmentation', location='data/raw', overwrite=False)
    print(f'Data ready at: {dataset.location}')

In [ ]:
# ── Cell 5: Load config and verify data ──────────────────────────────────────
import sys, os
sys.path.insert(0, '/content/mask2former/src')
os.chdir('/content/mask2former')

from config_loader import load_config
from data.dataset import build_dataloaders

# Colab: reduce workers to 2
cfg = load_config()

train_loader, val_loader = build_dataloaders(cfg)
print(f'Train: {len(train_loader.dataset)} samples')
print(f'Val:   {len(val_loader.dataset)} samples')

# Quick batch check
import torch
imgs, targets = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}')
print(f'Sample masks: {targets[0]["masks"].shape}')

In [ ]:
# ── Cell 6: Build model + verify freezing ────────────────────────────────────
import sys
sys.path.insert(0, '/content/mask2former/src')
from models.mask2former import build_model, set_phase

device = torch.device('cuda')
model = build_model(cfg).to(device)

# Show phase 1 trainable params
set_phase(model, phase=1)

# Sanity forward pass
model.eval()
with torch.no_grad():
    out = model(pixel_values=imgs.to(device))
print(f'Output keys: {list(out.keys())}')
print(f'Masks shape: {out.masks_queries_logits.shape}')
print(f'Logits shape: {out.class_queries_logits.shape}')

In [ ]:
# ── Cell 7: Check MLflow connection ─────────────────────────────────────────
import mlflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ['MLFLOW_EXPERIMENT_NAME'])

# Test connection with a quick log
with mlflow.start_run(run_name='connection-test') as run:
    mlflow.log_param('test', True)
    mlflow.log_metric('test_metric', 1.0)
    print(f'MLflow run ID: {run.info.run_id}')
    print(f'Tracking URI: {mlflow.get_tracking_uri()}')
print('MLflow connection OK')

In [ ]:
# ── Cell 8: Phase 1 — Transformer decoder only (epochs 0–15) ────────────────
# Backbone + pixel decoder frozen. Only the transformer decoder + class head train.
# Expected: loss drops fast (~48→25), mAP still 0 (scores below threshold — normal).
import sys
sys.path.insert(0, '/content/mask2former/src')
from train import train

best_map, run_id = train(cfg, stop_at_epoch=15)
print(f'\nPhase 1 done. best_mAP={best_map:.4f}  run_id={run_id}')

In [ ]:
# ── Cell 9: Phase 2 — Pixel decoder unfreezes (epochs 15–30) ────────────────
# Backbone still frozen. Feature pyramid now adapts to your lane data.
# Expected: loss continues dropping (~25→15), mAP may start appearing.
# Auto-resumes from last_checkpoint.pth and continues same MLflow run.

best_map, run_id = train(cfg, stop_at_epoch=30)
print(f'\nPhase 2 done. best_mAP={best_map:.4f}  run_id={run_id}')

In [ ]:
# ── Cell 10: Phase 3 — Full fine-tune (epochs 30–50) ────────────────────────
# All layers unfreeze with low LR. This is where real mAP gains happen.
# Expected: loss stabilizes (~10–15), mAP rises above 0.2+.
# Auto-resumes from last_checkpoint.pth and continues same MLflow run.

best_map, run_id = train(cfg, stop_at_epoch=50)
print(f'\nTraining complete! best_mAP={best_map:.4f}  run_id={run_id}')

In [ ]:
# ── Cell 11: (optional) Backup checkpoints to Google Drive ───────────────────
# Optional — use HF Hub (Cell 12) as primary artifact store after Colab restarts.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/mask2former-mlops/checkpoints'
os.makedirs(dst, exist_ok=True)

for ckpt in ['best_model.pth', 'phase1_final.pth', 'phase2_final.pth', 'phase3_final.pth', 'last_checkpoint.pth']:
    src = f'checkpoints/{ckpt}'
    if os.path.exists(src):
        shutil.copy(src, f'{dst}/{ckpt}')
        print(f'Copied: {ckpt}')

In [ ]:
# ── Cell 12: Push checkpoint to Hugging Face Hub ─────────────────────────────
from huggingface_hub import HfApi, create_repo

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = os.environ['HF_REPO_ID']

create_repo(repo_id, repo_type='model', private=True, exist_ok=True)

api.upload_file(
    path_or_fileobj='checkpoints/best_model.pth',
    path_in_repo='best_model.pth',
    repo_id=repo_id,
)
print(f'Model pushed to: https://huggingface.co/{repo_id}')

---
## Phase 2 — ONNX Export + INT8 Quantization

Run these cells after training (or with the test checkpoint pushed to HF Hub).  
**Requires:** `best_model.pth` in `checkpoints/`

In [ ]:
# ── Cell 14: Install ONNX dependencies ──────────────────────────────────────
!pip install -q onnx onnxruntime onnxruntime-tools onnxscript
print('ONNX dependencies installed')

In [ ]:
# ── Cell 15: Download checkpoint from HuggingFace Hub ───────────────────────
# Run this if you restarted Colab and need best_model.pth without re-training.
# Skip if checkpoints/ already has best_model.pth from current session.
from huggingface_hub import hf_hub_download
import os

os.makedirs('checkpoints', exist_ok=True)
hf_hub_download(
    repo_id=os.environ['HF_REPO_ID'],
    filename='best_model.pth',
    local_dir='checkpoints',
    token=os.environ['HF_TOKEN'],
)
print('Downloaded: checkpoints/best_model.pth')

In [ ]:
# ── Cell 16: Export PyTorch → ONNX fp32 (opset 16) ───────────────────────────
# Opset 16 uses the TorchScript exporter (dynamo exporter only at opset 18+).
# TorchScript exporter is required for static INT8 quantization compatibility.
import sys, os
sys.path.insert(0, '/content/mask2former/src')
os.chdir('/content/mask2former')

from export_onnx import export_to_onnx, verify_onnx

ckpt_path = 'checkpoints/best_model.pth'
fp32_path = 'checkpoints/mask2former_fp32.onnx'

export_to_onnx(
    checkpoint_path=ckpt_path,
    output_path=fp32_path,
    img_size=512,
    opset_version=16,   # TorchScript exporter — static quantization compatible
    device='cpu',
)

verify_onnx(ckpt_path, fp32_path, img_size=512)

In [ ]:
# ── Cell 17: Selective Static INT8 Quantization ──────────────────────────────
# Quantizes Conv/MatMul/Gemm only (Swin backbone heavy compute).
# Deformable attention stays fp32 (ONNX shape inference incompatibility).
#
# Type pairing (Q-ViT / FQ-ViT recommendation):
#   weight_type=QInt8   — weights can be negative
#   activation_type=QUInt8 — post-Softmax/GELU in [0,1], unsigned
#
# Calibrated on val set: domain-specific, not seen during training.
from quantize_int8 import quantize_int8, benchmark, _get_calibration_dir

int8_path = 'checkpoints/mask2former_int8.onnx'
cal_dir = _get_calibration_dir(cfg)

quantize_int8(
    fp32_onnx_path=fp32_path,
    int8_onnx_path=int8_path,
    calibration_dir=cal_dir,
    img_size=512,
    n_images=200,
)

print('\nBenchmark (CPU):')
t_fp32 = benchmark(fp32_path, img_size=512, n_runs=10)
t_int8 = benchmark(int8_path, img_size=512, n_runs=10)
print(f'Speedup: {t_fp32/t_int8:.1f}x')

In [ ]:
# ── Cell 18: Push ONNX models to HuggingFace Hub + backup to Drive ──────────
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
repo_id = os.environ['HF_REPO_ID']

for fname in ['mask2former_fp32.onnx', 'mask2former_int8.onnx']:
    local = f'checkpoints/{fname}'
    if os.path.exists(local):
        api.upload_file(
            path_or_fileobj=local,
            path_in_repo=fname,
            repo_id=repo_id,
        )
        size_mb = os.path.getsize(local) / 1e6
        print(f'Pushed: {fname} ({size_mb:.1f} MB) → https://huggingface.co/{repo_id}')

# Optional: backup ONNX to Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil, os
dst = '/content/drive/MyDrive/mask2former-mlops/checkpoints'
os.makedirs(dst, exist_ok=True)
for fname in ['mask2former_fp32.onnx', 'mask2former_int8.onnx']:
    src = f'checkpoints/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{dst}/{fname}')
        print(f'Backed up to Drive: {fname}')

print('\nONNX phase complete.')

---
## Phase 3 — Model Registry

Register the model in MLflow Model Registry (Staging) and push the model card to HF Hub.

In [ ]:
# ── Cell 20: Register model in MLflow + push model card to HF Hub ────────────
import sys, os, torch
sys.path.insert(0, '/content/mask2former/src')
os.chdir('/content/mask2former')

from register_model import register_in_mlflow, push_model_card

# Get run_id from the last checkpoint saved during training
last_ckpt = torch.load('checkpoints/last_checkpoint.pth', map_location='cpu')
run_id = last_ckpt.get('mlflow_run_id')

if run_id is None:
    # If last_checkpoint.pth isn't available, paste your run_id manually:
    run_id = 'PASTE_YOUR_RUN_ID_HERE'

print(f'Using MLflow run_id: {run_id}')

version = register_in_mlflow(
    run_id=run_id,
    checkpoint_path='checkpoints/best_model.pth',
    fp32_onnx_path='checkpoints/mask2former_fp32.onnx',
    int8_onnx_path='checkpoints/mask2former_int8.onnx',
)

# Push model card (README) to HF Hub
push_model_card(
    repo_id=os.environ['HF_REPO_ID'],
    hf_token=os.environ['HF_TOKEN'],
)

print(f'\nModel registry complete.')
print(f'Version {version} is in Staging.')
print(f'Run promote_to_production("{version}") after ROS2 validation.')